# `sesn_sedona` emulator examples

Post-process a MOSFiT **`sesn_sedona`** run: decode posterior velocity and
composition profiles from the latent parameters using the bundled
profile autoencoders.

Based on the SEDONA SESN grid of [Yadavalli et al. 2026](https://iopscience.iop.org/article/10.3847/1538-4357/ae32f8/meta) (ApJ 999 193).

For generic light-curve / corner plots, use `mosfit.ipynb` in this folder.

**Requirements:** `mosfit`, `torch`, `matplotlib`, `corner`, Jupyter.

Run from the `jupyter/` directory next to your run's `products/` folder.

This notebook reads a MOSFiT **`chain.json`** written with `-c`
(`[samples, param_names]`). Weight files ship under `mosfit/emulators/sesn_sedona/`
(`velocity_profile.pt`, `helium_profile.pt`, `nickel_profile.pt`, `opacity_profile.pt`),
overridable with `MOSFIT_EMULATOR_DATA`.


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

from mosfit.emulators.sesn_sedona.profiles import (
    get_element_profile,
    get_vel_profile,
    missing_weight_files,
    weights_dir,
)

plt.rcParams['font.family'] = 'serif'
plt.rcParams.update({'font.size': 14})

POS_COL = '#5DA5DA'
N_SAMPLES = 1000
BURN_FRAC = 0.5  # drop the first this fraction of steps as burn-in

# Point at a sesn_sedona chain.json (or set MOSFIT_CHAIN / leave None to auto-find).
CHAIN = Path('../products/chain_sesn_sedona_real.json')

REQUIRED_PARAMS = (
    'eta_vel', 'del_vel', 'min_vel',
    'eta_he', 'm_he', 'eta_ni', 'm_ni', 'eta_op', 'm_op',
)


def find_chain_path():
    if CHAIN is not None:
        path = Path(CHAIN).expanduser()
        if path.is_file():
            return path
        raise FileNotFoundError(f'CHAIN not found: {path}')
    env = os.environ.get('MOSFIT_CHAIN', '').strip()
    if env:
        path = Path(env).expanduser()
        if path.is_file():
            return path
    for rel in ('../products/chain.json', 'products/chain.json',
                '../products/chain.h5', 'products/chain.h5'):
        path = Path(rel)
        if path.is_file():
            return path
    raise FileNotFoundError(
        'chain.json not found. Run with notebook cwd=jupyter/, or export '
        'MOSFIT_CHAIN=/path/to/chain.json')


def load_chain(path):
    # Load MOSFiT chain.json as (samples[nwalkers, nsteps, nparam], names).
    path = Path(path)
    if path.suffix in ('.h5', '.hdf5'):
        import h5py
        with h5py.File(path, 'r') as hf:
            samples = np.asarray(hf['samples'][:], dtype=float)
            if samples.ndim == 4:
                samples = samples[0]
            names = [
                x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x)
                for x in hf['param_names'][:]
            ]
        return samples, names

    with open(path, encoding='utf-8') as f:
        raw = json.load(f)
    if not (isinstance(raw, list) and len(raw) == 2):
        raise ValueError(
            'Expected chain.json as [samples, param_names]; got {}'.format(
                type(raw)))
    samples = np.asarray(raw[0], dtype=float)
    names = list(raw[1])
    if samples.ndim == 4:
        samples = samples[0]
    if samples.ndim != 3:
        raise ValueError('Unexpected samples shape {}'.format(samples.shape))
    return samples, names


missing = missing_weight_files()
if missing:
    raise FileNotFoundError(
        'Missing SESN SEDONA profile weights in {}: {}. '
        'Place velocity/helium/nickel/opacity_profile.pt there, or set MOSFIT_EMULATOR_DATA.'.format(
            weights_dir(), ', '.join(missing)))

chain_path = find_chain_path()
print('Loading', chain_path, '...')
chain_samples, param_names = load_chain(chain_path)
print('Chain shape (nwalkers, nsteps, nparam) =', chain_samples.shape)
print('Parameters:', param_names)
print('Profile weights:', weights_dir())

missing_params = [k for k in REQUIRED_PARAMS if k not in param_names]
if missing_params:
    raise KeyError(
        'This chain is not from a sesn_sedona run (missing {}). '
        'Found: {}.'.format(', '.join(missing_params), ', '.join(param_names)))

# Drop burn-in, flatten walkers x steps, then draw N_SAMPLES rows.
nsteps = chain_samples.shape[1]
burn = int(nsteps * BURN_FRAC)
post = chain_samples[:, burn:, :].reshape(-1, chain_samples.shape[-1])
rng = np.random.default_rng(0)
draw = rng.choice(post.shape[0], size=min(N_SAMPLES, post.shape[0]), replace=False)
post = post[draw]
name_to_i = {n: i for i, n in enumerate(param_names)}


def row_params(row):
    return {n: float(row[name_to_i[n]]) for n in REQUIRED_PARAMS}


posterior = [row_params(row) for row in post]
print('Using {} posterior draws (burned first {:.0%} of steps)'.format(
    len(posterior), BURN_FRAC))


## Posterior velocity distribution

Draw samples from the chain (after burn-in), decode each velocity /
composition profile, and plot $\log_{10}$ velocity (km/s) versus enclosed
mass fraction.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='major', bottom=True, top=True, left=True, right=True,
               direction='in', length=25, labelsize=20)
ax.tick_params(which='minor', bottom=True, top=True, left=True, right=True,
               direction='in', length=10)

for p in posterior:
    # Parameter min_vel / del_vel are in units of 100 km/s (same as SESN SEDONA SED).
    vel = np.log10(
        get_vel_profile(
            D=p['eta_vel'],
            max_vel=p['del_vel'],
            min_vel=p['min_vel'],
        ) * 100.0)
    he = get_element_profile(
        R=p['eta_he'], total_mass=p['m_he'], element='2_4')
    ni = get_element_profile(
        R=p['eta_ni'], total_mass=p['m_ni'], element='28_56')
    op = get_element_profile(
        R=p['eta_op'], total_mass=p['m_op'], element='opacity')
    m_enc = np.cumsum(he + ni + op)
    ax.plot(m_enc / m_enc[-1], vel, color=POS_COL, alpha=0.1, zorder=10)

ax.set_xlabel('Fraction of Contained Mass', fontsize=30)
ax.set_ylabel(r'$\log_{10}$ [Velocity (km/s)]', fontsize=30)
ax.plot(np.nan, np.nan, linewidth=4, color=POS_COL, label='Posterior')
ax.legend(fontsize=20)
fig.subplots_adjust(left=0.14, right=0.965, bottom=0.135, top=0.97)
plt.show()
# fig.savefig('../products/velocity_profiles_posterior.pdf', dpi=300)


## Posterior mass fraction vs velocity

Plot He / Ni / opacity mass fractions versus velocity for the same chain draws.


In [ ]:
COLORS = {'He': '#5DA5DA', 'Ni': '#F17CB0', 'opacity': '#B2912F'}

fig, ax = plt.subplots(figsize=(10, 6))
for p in posterior:
    vel = get_vel_profile(
        D=p['eta_vel'],
        max_vel=p['del_vel'],
        min_vel=p['min_vel'],
    ) * 100.0  # km/s
    he = get_element_profile(
        R=p['eta_he'], total_mass=p['m_he'], element='2_4')
    ni = get_element_profile(
        R=p['eta_ni'], total_mass=p['m_ni'], element='28_56')
    op = get_element_profile(
        R=p['eta_op'], total_mass=p['m_op'], element='opacity')
    tot = he + ni + op
    tot = np.where(tot > 0, tot, np.nan)
    ax.plot(vel, he / tot, color=COLORS['He'], alpha=0.1, lw=1, zorder=10)
    ax.plot(vel, ni / tot, color=COLORS['Ni'], alpha=0.1, lw=1, zorder=10)
    ax.plot(vel, op / tot, color=COLORS['opacity'], alpha=0.1, lw=1, zorder=10)

for name, color in COLORS.items():
    ax.plot(np.nan, np.nan, color=color, lw=3, label=name)
ax.set_xlabel(r'Velocity (km/s)')
ax.set_ylabel('Mass fraction')
ax.set_yscale('log')
ax.set_ylim(1e-4, 1.0)
ax.legend(fontsize=14)
fig.tight_layout()
plt.show()
# fig.savefig('../products/mass_fraction_vs_velocity_posterior.pdf', dpi=300)


## Physical-parameter posteriors

Corner plot of the interpretable scales: $m_{\rm Ni}$, $m_{\rm He}$,
$m_{\rm op}$, $M_{\rm ej}=m_{\rm He}+m_{\rm Ni}+m_{\rm op}$, plus
`min_vel` and derived $v_{\max}\approx v_{\min}+\Delta v$
(in units of $100\,{\rm km\,s}^{-1}$).


In [ ]:
import corner

samples = np.array([
    [
        p['m_ni'],
        p['m_he'],
        p['m_op'],
        p['m_he'] + p['m_ni'] + p['m_op'],
        p['min_vel'],
        p['min_vel'] + p['del_vel'],
    ]
    for p in posterior
], dtype=float)
labels = [
    r'$m_{\rm Ni}\,(M_\odot)$',
    r'$m_{\rm He}\,(M_\odot)$',
    r'$m_{\rm op}\,(M_\odot)$',
    r'$M_{\rm ej}\,(M_\odot)$',
    r'$v_{\rm min}\,(100\,\mathrm{km\,s}^{-1})$',
    r'$v_{\rm max}\,(100\,\mathrm{km\,s}^{-1})$',
]

cfig = corner.corner(
    samples, labels=labels, quantiles=[0.16, 0.5, 0.84],
    show_titles=True, title_fmt='.3g')
plt.show()
# cfig.savefig('../products/sesn_sedona_physical_corner.pdf')
